# Tutorial: APEX for AIME (Math)
In this tutorial, we optimize GPT-4.1 Mini's Chain of Thought (`dspy.ChainOfThought`) for solving math problems (AIME) using the `dspy.APEX` optimizer. APEX performs targeted failure/success analyses, synthesizes hypotheses, and keeps the best prompts observed on the calibration set.

<details>
<summary>Recommended: Set up MLflow Autologging to understand what's happening under the hood.</summary>

### MLflow DSPy Integration

<a href="https://mlflow.org/">MLflow</a> is an LLMOps tool that natively integrates with DSPy and offers explainability and experiment tracking. MLflow's autologging capability automatically tracks progress of APEX optimization, as well as visualizes prompts and module executions as traces to understand DSPy's behavior better. You can set up MLflow easily by following the four steps below.

**Visualize module executions as traces**

![MLflow Trace](./mlflow-tracing-gepa-aime.png)

**Automatically track optimization progress and results**

![MLflow Tracking](./mlflow-tracking-gepa-aime-optimization.png)


**Setup MLflow**

1. Install MLflow

```bash
%pip install mlflow>=3.0.0
```

2. Start MLflow UI in a separate terminal
```bash
mlflow ui --port 5000 --backend-store-uri sqlite:///mlruns.db
```

3. Connect the notebook to MLflow
```python
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("DSPy")
```

4. Enable autologging.

```python
mlflow.dspy.autolog(
    # Log the optimization progress
    log_compiles=True,
    # Log the evaluation results
    log_evals=True,
    # Log traces from module executions
    log_traces=True,
)
```

To learn more about the integration, visit [MLflow DSPy Documentation](https://mlflow.org/docs/latest/llms/dspy/index.html) as well.
</details>

In [1]:
import os
import dspy
from dspy.adapters import JSONAdapter
import mlflow

# Set MLflow tracking URI
mlflow.set_tracking_uri("http://localhost:5005")
mlflow.set_experiment("dspy-experiments")

# Configure DSPy with MLflow logging
# dspy.configure(experimental=True)

# Enable autologging for DSPy
mlflow.dspy.autolog()

api_key = 'sk-12345' #input("Enter your OpenAI API key: ")
base_url = "https://nexus-master.lmndstaging.com"
model_prefix = "litellm_proxy"

student_lm = dspy.LM(
    model=f"{model_prefix}/openai/gpt-5-mini",
    api_key=api_key,
    base_url=base_url,
    reasoning_effort="minimal",
    temperature=0.0,
)
analysis_lm = dspy.LM(
    model=f"{model_prefix}/openai/gpt-5",
    api_key=api_key,
    base_url=base_url,
    reasoning_effort="minimal",
    temperature=1.0,
)

# APEX uses JSON adapters by default; exposing them makes customization explicit
analysis_adapter = JSONAdapter()
hypothesis_adapter = JSONAdapter()

n_threads = 50  # notebook thread budget used for evaluation and optimization

dspy.configure(lm=student_lm)
dspy.settings.configure(num_threads=n_threads)

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025/10/14 18:53:23 WARNING mlflow.utils.autologging_utils: MLflow dspy autologging is known to be compatible with 2.5.17 <= dspy, but the installed version is 3.0.4b1. If you encounter errors during autologging, try upgrading / downgrading dspy to a compatible version, or try upgrading MLflow.


### Loading the AIME dataset

The AIME exam consists of 2 problem sets of size 15 for each year. For this tutorial, we will use AIME problem sets from previous years (2022-2024) for optimization (amounting to total 3 years × 2 sets × 15 problems = 90 problems, split equally between train and validation sets), and test the performance on AIME 2025 (2 sets × 15 problems = 30 problems). Since AIME 2025 is a small set, we repeat it 5 times for statistical stability in evaluation.

In [2]:
from datasets import load_dataset
import random


def init_dataset():
    train_split = load_dataset("AI-MO/aimo-validation-aime")['train']
    train_split = [
        dspy.Example({
            "problem": x['problem'],
            "solution": x['solution'],
            "answer": x['answer'],
        }).with_inputs("problem")
        for x in train_split
    ]
    random.Random(0).shuffle(train_split)
    tot_num = len(train_split)

    test_split = load_dataset("MathArena/aime_2025")['train']
    test_split = [
        dspy.Example({
            "problem": x['problem'],
            "answer": x['answer'],
        }).with_inputs("problem")
        for x in test_split
    ]

    train_set = train_split[: int(0.5 * tot_num)]
    val_set = train_split[int(0.5 * tot_num):]
    test_set = test_split * 5

    return train_set, val_set, test_set

In [3]:
train_set, val_set, test_set = init_dataset()

len(train_set), len(val_set), len(test_set)

(45, 45, 150)

Let's view an example task input

In [4]:
print("Problem:")
print(train_set[0]['problem'])
print("\n\nSolution:")
print(train_set[0]['solution'])
print("\n\nAnswer:")
print(train_set[0]['answer'])

Problem:
In isosceles trapezoid $ABCD$, parallel bases $\overline{AB}$ and $\overline{CD}$ have lengths $500$ and $650$, respectively, and $AD=BC=333$. The angle bisectors of $\angle{A}$ and $\angle{D}$ meet at $P$, and the angle bisectors of $\angle{B}$ and $\angle{C}$ meet at $Q$. Find $PQ$.


Solution:
We have the following diagram:

Let $X$ and $W$ be the points where $AP$ and $BQ$ extend to meet $CD$, and $YZ$ be the height of $\triangle AZB$. As proven in Solution 2, triangles $APD$ and $DPW$ are congruent right triangles. Therefore, $AD = DW = 333$. We can apply this logic to triangles $BCQ$ and $XCQ$ as well, giving us $BC = CX = 333$. Since $CD = 650$, $XW = DW + CX - CD = 16$.
Additionally, we can see that $\triangle XZW$ is similar to $\triangle PQZ$ and $\triangle AZB$. We know that $\frac{XW}{AB} = \frac{16}{500}$. So, we can say that the height of the triangle $AZB$ is $500u$ while the height of the triangle $XZW$ is $16u$. After that, we can figure out the distance from 

### Let's define the program: A simple `dspy.ChainOfThought`

In [5]:
class GenerateResponse(dspy.Signature):
    """Solve the problem and provide the answer in the correct format."""
    problem = dspy.InputField()
    answer = dspy.OutputField()


program = dspy.ChainOfThought(GenerateResponse)

### Defining the evaluation metric
We simply check exact match between the predicted answer and the correct answer.

In [6]:
def metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    try:
        llm_answer = int(prediction.answer)
    except ValueError:
        return 0
    return int(correct_answer == llm_answer)

### Evaluating unoptimized Chain Of Thought

We evaluate with the thread budget defined above and tolerate up to `len(test_set)` transient errors so the run completes even on constrained proxies. If your provider enforces stricter limits, lower `n_threads` or tighten `max_errors`.

In [7]:
# Removed max_errors configuration since there should be no errors
eval_kwargs = dict(
    num_threads=n_threads,
    display_progress=True,
    display_table=5,
    provide_traceback=False,
)

evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    **eval_kwargs,
)

baseline_result = evaluate(program)
baseline_result.score

Average Metric: 80.00 / 150 (53.3%): 100%|██████████| 150/150 [00:04<00:00, 34.89it/s]

2025/10/14 18:53:32 INFO dspy.evaluate.evaluate: Average Metric: 80 / 150 (53.3%)


,problem,example_answer,reasoning,pred_answer,metric
0,Find the sum of all integer bases $b>9$ for which $17_b$ is a divi...,70,We interpret 17_b = b+7 and 97_b = 9b+7. We need b+7 to divide 9b+...,70,✔️ [1]
1,"On $\triangle ABC$ points $A, D, E$, and $B$ lie in that order on ...",588,"Set up affine coordinates with A=(0,0), B=(1,0), C=(0,1). Points o...",441,✔️ [0]
2,The 9 members of a baseball team went to an ice-cream parlor after...,16,"We must count assignments of 3 labeled flavors (C, V, S) to 9 dist...",16,✔️ [1]
3,"Find the number of ordered pairs $(x,y)$, where both $x$ and $y$ a...",117,"We need integer solutions (x,y) in [-100,100] satisfying 12x^2 - x...",117,✔️ [1]
4,There are $8!= 40320$ eight-digit positive integers that use each ...,279,Divisibility by 22 means divisible by 2 and 11. Units digit must b...,279,✔️ [1]


53.33

[Trace(trace_id=tr-4cfdcd178635b6f49f180b65cace7270), Trace(trace_id=tr-7b344ddc70882da588fbba9dc42f782b), Trace(trace_id=tr-c8b894808f5a5620f748d0ff66f81dcd), Trace(trace_id=tr-dddf7e09c547d5e9bb4a578a80a80f01), Trace(trace_id=tr-4ed37ad9bb81653c70b0c2898a7b35d7), Trace(trace_id=tr-97f8caffe2295e02ed12a39da56ecbe5), Trace(trace_id=tr-8f00c7abb5392a03d3a6c85d225ebc2a), Trace(trace_id=tr-fe1fbdf9286f8838081e5b1a5c4e0af3), Trace(trace_id=tr-c241d4b65ce7d8b746c52c8fae5cfbbf), Trace(trace_id=tr-a909293e9d5bf0fc5cea6d872a817541)]

### Augmenting the metric for APEX
APEX benefits from feedback about why predictions fail. We extend the metric to provide textual guidance (and optional worked solutions) that the optimizer can feed into its failure and success analyses.

In [8]:
def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    written_solution = example.get('solution', '')
    try:
        llm_answer = int(prediction.answer)
    except ValueError:
        feedback_text = (
            f"The final answer must be a valid integer and nothing else. You responded with '{prediction.answer}', which couldn't be parsed as an integer."
        )
        feedback_text += f" The correct answer is '{correct_answer}'."
        if written_solution:
            feedback_text += (
                f" Here's the full step-by-step solution:\n{written_solution}\n\n"
                "Reflect on this solution and ensure your final answer is a valid integer when you attempt similar problems."
            )
        return dspy.Prediction(score=0, feedback=feedback_text)

    score = int(correct_answer == llm_answer)
    if score == 1:
        feedback_text = f"Your answer is correct. The correct answer is '{correct_answer}'."
    else:
        feedback_text = f"Your answer is incorrect. The correct answer is '{correct_answer}'."

    if written_solution:
        feedback_text += (
            f" Here's the full step-by-step solution:\n{written_solution}\n\n"
            "Use it to identify the mistakes in your reasoning before trying again."
        )

    return dspy.Prediction(score=score, feedback=feedback_text)

### Optimize the program with `dspy.APEX`

APEX runs targeted analyses over failure and success cases, proposes hypotheses with complete prompt updates, and keeps the best candidate on the calibration set. We limit the budget to a few iterations to keep the tutorial runtime manageable. Use `verbosity` to control logging (`"none"`, `"normal"`, or `"high"`) and `num_threads` to parallelize execution.

In [9]:
from dspy.teleprompt.apex_optimizer import APEX

# Fixed configuration for parallel execution with enhanced visibility
optimizer = APEX(
    metric=metric_with_feedback,
    analysis_lm=analysis_lm,        # Required parameter
    hypothesis_lm=analysis_lm,       # Optional, defaults to analysis_lm if not provided
    analysis_adapter=analysis_adapter,
    hypothesis_adapter=hypothesis_adapter,
    max_iterations=1,
    num_hypotheses=1,
    num_eval_runs=1,
    train_sample=20,
    success_threshold=1.0,
    convergence_patience=5,
    num_threads=n_threads,           # Using n_threads=50 from configuration
    verbosity="high",                # Enhanced visibility into the optimization process
    seed=42,
)

optimized_program = optimizer.compile(
    student=program,
    trainset=train_set,
    valset=val_set,
)

2025/10/14 18:53:33 INFO dspy.teleprompt.apex_optimizer: APEX: running with num_threads=50
2025/10/14 18:53:33 INFO dspy.teleprompt.apex_optimizer: APEX: Configuration - max_iterations=1, num_hypotheses=1, success_threshold=1.00, convergence_patience=5
2025/10/14 18:53:33 INFO dspy.teleprompt.apex_optimizer: APEX: Using seed=42 for reproducibility
2025/10/14 18:53:33 INFO dspy.teleprompt.apex_optimizer: APEX: Evaluating initial baseline on validation set


Processed 45 / 45 examples: 100%|██████████| 45/45 [00:00<00:00, 446.91it/s]

2025/10/14 18:53:34 INFO dspy.teleprompt.apex_optimizer: APEX: Initial baseline score=0.5111
2025/10/14 18:53:34 INFO dspy.teleprompt.apex_optimizer: APEX: iteration 1 started (train sample=20, val size=45)
2025/10/14 18:53:34 INFO dspy.teleprompt.apex_optimizer: APEX: Sampled 20 training examples from 45 total



Processed 20 / 20 examples: 100%|██████████| 20/20 [00:00<00:00, 107.57it/s]

2025/10/14 18:53:34 INFO dspy.teleprompt.apex_optimizer: APEX: Train evaluation complete - 6 failures, 14 successes


...

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='[[ ## an...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


Processed 23 / 45 examples:  51%|█████     | 23/45 [00:57<01:15,  3.43s/it]

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='{\n  "re...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


Processed 45 / 45 examples: : 47it [03:52,  4.95s/it]                      

2025/10/14 18:58:41 INFO dspy.teleprompt.apex_optimizer: APEX: iteration 1 hypothesis score=0.6444
2025/10/14 18:58:41 INFO dspy.teleprompt.apex_optimizer: APEX: hypothesis details → {'observation': 'Failures cluster around the single predictor making premature guesses, misreading constraints, and outputting wrong numeric values due to skipped derivations and missing explicit problem-structure steps. Successes show that when the solver explicitly models constraints, builds formulas, and computes step-by-step, results are correct.', 'fixable_root_causes': ['Ambiguous instruction leading to guessing without derivation', 'Missing explicit constraints and validation checks before final answer', 'Lack of required step-by-step algebraic or combinatorial computation', 'No final consistency check against problem conditions', 'Failure to state and use variable definitions and known formulas'], 'non_fixable_root_causes': [], 'impact_score': 0.82, 'generalizability_score': 0.86, 'strategy': 'Intr

### Inspect the APEX-optimized prompt

In [10]:
print(optimized_program.predict.signature.instructions)

You are a competition math solver. Solve the problem rigorously and return only the final numeric answer in the required format. Do not guess. Always derive and verify.

Solution protocol (follow all steps):
1) Parse and restate: Identify what is asked (target quantity) and list given constraints and definitions. Introduce variables with clear meanings.
2) Plan: Choose a method suited to the domain (algebraic identities, combinatorics counting with inclusion–exclusion, geometry with similarity/ratios, graph/region formulas, etc.). Cite any standard formula you will use (e.g., C(n,2)C(m,2)+mn−1 for bounded regions formed by m vertical and n horizontal lines; palindrome structure; symmetric polynomial identities; Gram matrix determinant for volumes).
3) Derive: Execute the plan step-by-step. Show key intermediate expressions (not verbose prose). Carry out necessary case analyses fully when counting. For geometry, set up ratios or coordinates; for counting, enumerate cases and subtract fo

Trace(trace_id=tr-57aa38477256d5ebe6a26dceabc057b7)

### Evaluating the Chain Of Thought optimized with APEX

In [11]:
evaluate(optimized_program)

Average Metric: 2.00 / 2 (100.0%):   1%|          | 1/150 [00:12<29:41, 11.96s/it]

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='[[ ## re...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='[[ ## re...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpec

Average Metric: 3.00 / 3 (100.0%):   2%|▏         | 3/150 [00:13<08:41,  3.55s/it]

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='[[ ## re...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


Average Metric: 4.00 / 4 (100.0%):   3%|▎         | 4/150 [00:15<07:46,  3.20s/it]

...

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='{\n  "re...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
2025/10/14 19:02:15 INFO dspy.evaluate.evaluate: Average Metric: 95 / 150 (63.3%)


,problem,example_answer,reasoning,pred_answer,metric
0,Find the sum of all integer bases $b>9$ for which $17_b$ is a divi...,70,1) Parse and restate: - We are asked to find sum of all integer ba...,70,✔️ [1]
1,"On $\triangle ABC$ points $A, D, E$, and $B$ lie in that order on ...",588,"1) Parse and restate: - Triangle ABC with A--D--E--B on AB, length...",588,✔️ [1]
2,The 9 members of a baseball team went to an ice-cream parlor after...,16,1) Parse and restate: - There are 9 distinct players. Each chooses...,16,✔️ [1]
3,"Find the number of ordered pairs $(x,y)$, where both $x$ and $y$ a...",117,1) Parse and restate: We seek the number of ordered integer pairs ...,117,✔️ [1]
4,There are $8!= 40320$ eight-digit positive integers that use each ...,279,1) Parse and restate: - We consider all 8-digit permutations of di...,279,✔️ [1]


EvaluationResult(score=63.33, results=<list of 150 results>)

[Trace(trace_id=tr-e5deda72a41355e90ecc9e1cb4e25945), Trace(trace_id=tr-37602be62283376eae2bed90de62be84), Trace(trace_id=tr-69064799399fe1770be869aa7dccc5ee), Trace(trace_id=tr-9bff023b3bace4f86ac58a92719438af), Trace(trace_id=tr-f3d6a61869d2e9a3bac9a3c5be4567f4), Trace(trace_id=tr-0c8dc903cae334d9e91f49e807ede1b9), Trace(trace_id=tr-9e5a3db66202956731613ab76d20ced7), Trace(trace_id=tr-47903ef023a54336f3538c245f13a7aa), Trace(trace_id=tr-b810af6fde8f5d94276f5bd62fdc1ec2), Trace(trace_id=tr-747d4d0db8a675e142db7b9bdb0e6588)]

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='{\n  "re...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='{\n  "re...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpec

APEX typically improves the GPT-4.1 Mini's performance on AIME 2025 by leveraging targeted analyses while keeping the overall evaluation flow identical to the GEPA tutorial.